In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
RUTA_POR_DEFECTO = "UTILITY.xls"


def elegir_archivo():
    ruta = input(
        f"Ruta del archivo Excel (Enter para usar '{RUTA_POR_DEFECTO}'): "
    ).strip().strip('"')
    return ruta if ruta else RUTA_POR_DEFECTO


def elegir_columna(df):
    columnas_numericas = df.select_dtypes(include=[np.number]).columns.tolist()
    if not columnas_numericas:
        raise ValueError("El archivo no tiene columnas numéricas.")

    print("\nColumnas numéricas disponibles:")
    for i, col in enumerate(columnas_numericas, start=1):
        print(f"  {i}. {col}")

    while True:
        seleccion = input("\nElegí el número de la columna a analizar: ").strip()
        if seleccion.isdigit() and 1 <= int(seleccion) <= len(columnas_numericas):
            return columnas_numericas[int(seleccion) - 1]
        print("Opción inválida, intentá de nuevo.")


def resumen_cinco_numeros(data):
    minimo = data.min()
    q1 = np.percentile(data, 25)
    mediana = np.median(data)
    q3 = np.percentile(data, 75)
    maximo = data.max()
    return minimo, q1, mediana, q3, maximo


def calcular_outliers_tukey(data):
    q1 = np.percentile(data, 25)
    q3 = np.percentile(data, 75)
    iqr = q3 - q1
    limite_inferior = q1 - 1.5 * iqr
    limite_superior = q3 + 1.5 * iqr
    outliers = data[(data < limite_inferior) | (data > limite_superior)]
    return outliers, limite_inferior, limite_superior


def graficar(data, columna):
    minimo, q1, mediana, q3, maximo = resumen_cinco_numeros(data)
    outliers, lim_inf, lim_sup = calcular_outliers_tukey(data)

    fig, axes = plt.subplots(1, 2, figsize=(12, 6))
    fig.suptitle(f"Análisis de la columna: {columna}", fontsize=14, fontweight="bold")

    # Boxplot "completo": los bigotes se estiran hasta el mínimo y el máximo,
    # por lo que los outliers quedan incluidos dentro de los bigotes
    # (no se marcan como puntos aparte).
    axes[0].boxplot(
        data, vert=True, whis=[0, 100], patch_artist=True,
        boxprops=dict(facecolor="#8ecae6"),
    )
    axes[0].set_title("Boxplot con outliers\n(dentro de los bigotes)")
    axes[0].set_ylabel(columna)

    # Boxplot de Tukey: los bigotes llegan hasta 1.5*IQR y los outliers
    # se muestran como puntos separados, por fuera de los bigotes.
    axes[1].boxplot(
        data, vert=True, whis=1.5, showfliers=True, patch_artist=True,
        boxprops=dict(facecolor="#ffb703"),
        flierprops=dict(marker="o", markerfacecolor="red", markersize=6, alpha=0.6),
    )
    axes[1].set_title("Boxplot de Tukey\n(outliers fuera de los bigotes)")
    axes[1].set_ylabel(columna)

    resumen_texto = (
        f"Resumen de los cinco números — {columna}\n"
        f"Mínimo: {minimo:.2f}    Q1: {q1:.2f}    Mediana: {mediana:.2f}    "
        f"Q3: {q3:.2f}    Máximo: {maximo:.2f}\n"
        f"Límites de Tukey (1.5×IQR): [{lim_inf:.2f}, {lim_sup:.2f}]    "
        f"Outliers detectados: {len(outliers)}"
    )

    fig.text(
        0.5, 0.02, resumen_texto, ha="center", fontsize=10,
        bbox=dict(boxstyle="round", facecolor="whitesmoke", edgecolor="gray"),
    )

    plt.tight_layout(rect=[0, 0.10, 1, 0.95])
    plt.show()

    print("\n" + resumen_texto)
    if len(outliers) > 0:
        print("\nValores outliers:")
        print(outliers.to_string(index=False))


def main():
    ruta = elegir_archivo()
    df = pd.read_excel(ruta)
    columna = elegir_columna(df)
    data = df[columna].dropna()
    graficar(data, columna)


if __name__ == "__main__":
    main()